In [ ]:
# Parameters
slice_name = "mfportal_test"


#### Measurement Framework Library
# MFPortal API Test: Add Meas Node (FABNetv6)

Tests `MFPortal.add_meas_node()` from `mflib/mfportal.py` — the
rewrite of `MFLib.addMeasNode()` that attaches the meas node via a
FABNetv6 (IPv6) network instead of a per-site FABNetv4 network.

Follows the same structure as [prepare-a-slice.ipynb](./prepare-a-slice.ipynb):
build a 3-node experiment topology, add the meas node, submit. The only
change from that notebook is which method adds the meas node.

## General Imports

In [ ]:
import os
import json
import traceback

## Import MFPortal

This imports `MFPortal` from `mflib.mfportal`. If you have trouble
importing `mflib`, see [Install MFLib](./mflib_install.ipynb).

In [ ]:
import mflib
print(f"MFLib version  {mflib.__version__} ")

from mflib.mfportal import MFPortal

## Setup Experiment Slice

### Import fablib

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()
    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")

### Set Slice Information

In [ ]:
%%time

# Sites without PTP + NCSA where meas node is located + problem sites
[site1, site2, site3] = fablib.get_random_sites(
    count=3,
    avoid=["DALL","GPN","LBNL","RENC","SALT","TACC","UKY","WASH","NCSA",
           "LOSA","GATECH","INDI","MAX","MASS","NEWY","SRI","UCSD"],
)

node1_name = 'Node1'
node2_name = 'Node2'
node3_name = 'Node3'

network1_name = 'net1'
network2_name = 'net2'
network3_name = 'net3'

node1_nic_name = 'nic1'
node2_nic_name = 'nic2'
node3_nic_name = 'nic3'

print(f"Setting up slice {slice_name}")
print(f"Using sites {site1}, {site2}, {site3}")

### Create Experiment Topology

In [ ]:
try:
    # Create Slice
    slice = fablib.new_slice(name=slice_name)

    # Node1
    node1 = slice.add_node(name=node1_name, site=site1)
    iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]

    # Node2
    node2 = slice.add_node(name=node2_name, site=site2)
    iface2 = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

    # Node3
    node3 = slice.add_node(name=node3_name, site=site3)
    iface3 = node3.add_component(model='NIC_Basic', name=node3_nic_name).get_interfaces()[0]

    # Networks
    net1 = slice.add_l3network(name=network1_name, interfaces=[iface1], type='IPv4')
    net2 = slice.add_l3network(name=network2_name, interfaces=[iface2], type='IPv4')
    net3 = slice.add_l3network(name=network3_name, interfaces=[iface3], type='IPv4')

    print(f"Slice Topology Done.")
except Exception as e:
    print(f"Exception: {e}")

### Add measurement node to slice topology (via MFPortal)

In `prepare-a-slice.ipynb` this step is `MFLib.addMeasNode(slice)`, which
attaches the meas node with a per-site FABNetv4 network (`l3_meas_net_<site>`).

Here we call `MFPortal.add_meas_node(slice)` instead — the method under test —
which attaches the meas node via a single FABNetv6 network
(`MFPortal.MEAS_NETWORK_NAME`, `"meas-net6"`) instead.

In [ ]:
# Add measurement node to topology using the new MFPortal method.
MFPortal.add_meas_node(slice, disk_gb=100, image='default_ubuntu_24')
print("Done")

### Submit the Slice

In [ ]:
%%time
try:
    # Submit Slice Request
    print(f'Submitting the new slice, "{slice_name}"...')
    slice.submit()
    print(f'{slice_name} creation done.')

except Exception as e:
    print(f"Slice Fail: {e}")
    traceback.print_exc()

### Verify the meas node

A quick smoke test of two more `MFPortal` methods against the
now-submitted slice: `collect_node_info()` and `assign_static_fabnet6_ip()`.
The latter should show a real FABNetv6 subnet/gateway/IP for the meas node's
`meas-nic6` interface.

In [ ]:
node_info = MFPortal.collect_node_info(slice, meas_node_name=MFPortal.MEAS_NODE_NAME)
node = node_info['node']
node_info

In [ ]:
fabnet_info = MFPortal.assign_static_fabnet6_ip(
    slice, node, meas_network_name=MFPortal.MEAS_NETWORK_NAME
)
fabnet_info

-----
# Slice Setup Is Complete

The slice now has the meas node attached via FABNetv6 using
`MFPortal.add_meas_node()`. Compare this flow against
[prepare-a-slice.ipynb](./prepare-a-slice.ipynb) (which uses
`MFLib.addMeasNode()`) to see how the two approaches differ. The prior
version of `MFPortal` (moved wholesale out of `mflib.py`, before this
notebook-derived rewrite) is kept at `mflib/first-draft-mfportal.py` for
reference.

-----